In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from preprocessing.embedding import (
    EMBEDDING_DIM,
    check_reviews,
    embeddings_to_dataframe,
    generate_embeddings,
    load_model,
    validate_embeddings,
)

In [2]:
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "IMDB_Dataset_menor_encoded.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "IMDB_menor_embeddings.npy"
CSV_OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "IMDB_menor_embeddings.csv"

print("Entrada:", INPUT_FILE)
print("Saída:", OUTPUT_FILE)

Entrada: C:\Users\kaick\PycharmProjects\Projeto-IA-classificacaoSentimentosFilme\data\processed\IMDB_Dataset_menor_encoded.csv
Saída: C:\Users\kaick\PycharmProjects\Projeto-IA-classificacaoSentimentosFilme\data\processed\IMDB_menor_embeddings.npy


In [3]:
df = pd.read_csv(INPUT_FILE)

print("Quantidade de linhas:", len(df))
print("Quantidade de colunas:", len(df.columns))

df.head()

Quantidade de linhas: 2000
Quantidade de colunas: 2


,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production. the filming tec...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there's a family where a little boy ...,0
4,"petter mattei's ""love in the time of money"" is...",1


In [4]:
print("Valores ausentes:", df["review"].isnull().sum())
print("Valores vazios:", (df["review"].astype(str).str.strip() == "").sum())

check_reviews(df)

Valores ausentes: 0
Valores vazios: 0


In [5]:
model = load_model()

if hasattr(model, "get_embedding_dimension"):
    dimensao = model.get_embedding_dimension()
else:
    dimensao = model.get_sentence_embedding_dimension()

print("Dimensão dos vetores:", dimensao)
print("Limite de tokens:", model.max_seq_length)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dimensão dos vetores: 384
Limite de tokens: 256


In [6]:
exemplos = [
    "this movie was amazing, i loved it",
    "one of the best films i have ever seen",
    "terrible movie, a complete waste of time",
]

vetores = generate_embeddings(exemplos, model=model)
normas = np.linalg.norm(vetores, axis=1, keepdims=True)
similaridade = (vetores / normas) @ (vetores / normas).T

pd.DataFrame(similaridade, index=exemplos, columns=["1", "2", "3"]).round(3)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,1,2,3
"this movie was amazing, i loved it",1.000,0.599,0.467
one of the best films i have ever seen,0.599,1.000,0.497
"terrible movie, a complete waste of time",0.467,0.497,1.000


In [7]:
embeddings = generate_embeddings(df["review"].tolist(), model=model)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [8]:
print("Formato da matriz:", embeddings.shape)
print("Esperado:", (len(df), EMBEDDING_DIM))
print("Valores NaN:", np.isnan(embeddings).sum())

validate_embeddings(embeddings, len(df))

Formato da matriz: (2000, 384)
Esperado: (2000, 384)
Valores NaN: 0


In [9]:
for i in [0, len(df) // 2, len(df) - 1]:
    individual = generate_embeddings([df.loc[i, "review"]], model=model)[0]
    print(f"Linha {i}:", np.allclose(embeddings[i], individual, atol=1e-5))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Linha 0: True


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Linha 1000: True


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Linha 1999: True


In [10]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_FILE, embeddings)
print(f"Embeddings salvos em:\n{OUTPUT_FILE}")

Embeddings salvos em:
C:\Users\kaick\PycharmProjects\Projeto-IA-classificacaoSentimentosFilme\data\processed\IMDB_menor_embeddings.npy


In [11]:
embeddings_teste = np.load(OUTPUT_FILE)

print("Formato:", embeddings_teste.shape)
print("Iguais ao gerado:", np.array_equal(embeddings, embeddings_teste))

Formato: (2000, 384)
Iguais ao gerado: True


In [12]:
df_embeddings = embeddings_to_dataframe(embeddings, df["sentiment"])

df_embeddings.to_csv(CSV_OUTPUT_FILE, index=False)

print(f"Tabela salva em:\n{CSV_OUTPUT_FILE}")
print("Formato:", df_embeddings.shape)

df_embeddings.head()

Tabela salva em:
C:\Users\kaick\PycharmProjects\Projeto-IA-classificacaoSentimentosFilme\data\processed\IMDB_menor_embeddings.csv
Formato: (2000, 385)


,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,...,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,sentiment
0,0.028352,0.058333,-0.069887,0.068288,0.080217,0.069166,0.040935,0.024101,0.073692,-0.060215,...,-0.078145,-0.024826,0.016664,0.013412,0.020879,0.065421,-0.025376,-0.050503,-0.025903,1
1,0.008254,0.065314,-0.004735,-0.051875,0.020606,0.084680,0.021552,0.006631,-0.015105,-0.015956,...,0.036724,0.038877,0.004208,-0.072306,0.010558,0.030873,0.010355,-0.050139,0.005361,1
2,0.014115,-0.076151,0.000104,0.013555,0.075392,0.070780,0.089844,0.067970,-0.018511,0.013720,...,-0.083469,-0.130850,0.057835,-0.062770,0.041212,0.055430,-0.030518,-0.000004,-0.083408,1
3,-0.051149,0.010064,-0.058461,-0.029110,0.056663,0.086137,0.018702,-0.010250,0.084562,-0.042020,...,-0.036892,0.000891,-0.016270,-0.003118,0.024032,0.069954,-0.015741,0.014551,0.015636,0
4,-0.040961,-0.015201,-0.030476,-0.034416,-0.023005,0.049733,0.055261,0.000040,0.111896,-0.046311,...,-0.009982,-0.027761,0.108200,-0.008198,-0.013424,0.011123,-0.027719,0.008909,0.026550,1
